In [1]:
import re
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

class Agent1TopicModeling:
    """
    Agent 1: Extraction & Topic Modeling Agent
    ------------------------------------------
    A modular AI agent that processes scientific abstracts on energy catalysis and critical minerals.
    - Preprocesses text and applies regex entity extraction for materials and reactions.
    - Constructs TF-IDF sparse vector representations (AIMA Ch. 22, Xiao Ch. 7).
    - Performs automated parameter sweeps across k-clusters (AIMA Ch. 20, Xiao Ch. 8).
    - Generates diagnostic evaluation plots (Silhouette Score curve & Elbow plot).
    - Assigns cluster IDs, generates keyword-driven topic labels, and exports enriched datasets.
    """
    
    def __init__(self, data_path="openalex_catalysts_3000.csv"):
        self.data_path = data_path
        self.df = None
        self.vectorizer = None
        self.tfidf_matrix = None
        self.best_k = None
        self.kmeans_model = None
        self.topic_labels = {}
        self.silhouette_scores = {}
        self.inertias = {}
        
        # Define regex dictionary for domain-specific entity extraction
        self.material_patterns = {
            'Fe-N-C / M-N-C': r'\b(fe-n-c|co-n-c|m-n-c|single-atom fe|iron-nitrogen-carbon)\b',
            'NiFe-LDH / Hydroxides': r'\b(nife-ldh|layered double hydroxide|transition metal phosphide|selenide)\b',
            'MoS2 / Dichalcogenides': r'\b(mos2|ws2|molybdenum disulfide|dichalcogenide)\b',
            'Copper Nanostructures': r'\b(cu nanostructures|mof-derived cu|bimetallic cu|cu-n-c)\b',
            'PGM-Free General': r'\b(pgm-free|non-precious|earth-abundant|noble-metal-free)\b'
        }
        
        self.reaction_patterns = {
            'ORR (Oxygen Reduction)': r'\b(orr|oxygen reduction|fuel cell|pemfc|zinc-air)\b',
            'OER (Oxygen Evolution)': r'\b(oer|oxygen evolution|water splitting|aemfc|electrolyzer)\b',
            'HER (Hydrogen Evolution)': r'\b(her|hydrogen evolution|water electrolysis|acidic her)\b',
            'CO2RR (CO2 Reduction)': r'\b(co2rr|co2 reduction|carbon dioxide reduction|c2\+ hydrocarbon)\b',
            'NRR (Ammonia Synthesis)': r'\b(nrr|nitrogen reduction|ammonia synthesis|nitrogen fixation)\b'
        }

    def load_data(self):
        """Loads dataset from CSV, Excel, or JSON."""
        print(f"[Agent 1] Loading dataset from: {self.data_path}")
        if self.data_path.endswith('.csv'):
            self.df = pd.read_csv(self.data_path)
        elif self.data_path.endswith('.xlsx'):
            self.df = pd.read_excel(self.data_path)
        elif self.data_path.endswith('.json'):
            with open(self.data_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            self.df = pd.DataFrame(data)
        else:
            raise ValueError("Unsupported file format. Please provide .csv, .xlsx, or .json.")
        
        print(f"[Agent 1] Successfully loaded {len(self.df)} records.")
        return self.df

    def extract_entities(self, text):
        """Applies domain regex rules to extract catalyst materials and energy reactions."""
        text_lower = str(text).lower()
        
        extracted_mats = []
        for cat, pattern in self.material_patterns.items():
            if re.search(pattern, text_lower):
                extracted_mats.append(cat)
                
        extracted_rxns = []
        for cat, pattern in self.reaction_patterns.items():
            if re.search(pattern, text_lower):
                extracted_rxns.append(cat)
                
        return "; ".join(extracted_mats) if extracted_mats else "General Catalysts", \
               "; ".join(extracted_rxns) if extracted_rxns else "General Electrocatalysis"

    def preprocess_and_vectorize(self, max_features=1000):
        """
        Preprocesses text and constructs TF-IDF feature matrix.
        Custom stop words filter out generic paper prose.
        """
        print("[Agent 1] Extracting domain entities and constructing TF-IDF vector matrix...")
        
        # Entity extraction
        entity_results = self.df['abstract'].apply(self.extract_entities)
        self.df['extracted_materials'] = [res[0] for res in entity_results]
        self.df['extracted_reactions'] = [res[1] for res in entity_results]
        
        # Custom stop words for scientific prose
        custom_stop_words = [
            'catalyst', 'catalysts', 'catalytic', 'electrocatalyst', 'electrocatalysts',
            'performance', 'demonstrated', 'exhibited', 'herein', 'activity', 'synthesis',
            'synthesized', 'developed', 'study', 'results', 'showed', 'reported', 'using',
            'high', 'electrodes', 'density', 'functional', 'theory', 'dft', 'calculations'
        ]
        
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            stop_words='english'
        )
        
        # Combine title and abstract for richer vector representation
        text_corpus = (self.df['title'].fillna('') + " " + self.df['abstract'].fillna('')).str.lower()
        
        # Remove custom stop words
        for word in custom_stop_words:
            text_corpus = text_corpus.str.replace(rf'\b{word}\b', '', regex=True)
            
        self.tfidf_matrix = self.vectorizer.fit_transform(text_corpus)
        print(f"[Agent 1] Feature matrix constructed with shape: {self.tfidf_matrix.shape}")

    def sweep_hyperparameters(self, min_k=2, max_k=10):
        """
        Sweeps cluster counts k in [min_k, max_k].
        Calculates Silhouette Scores and Inertia (Elbow method).
        Selects optimal k programmatically based on max Silhouette score.
        """
        print(f"[Agent 1] Running hyperparameter sweep over k in [{min_k}, {max_k}]...")
        
        best_score = -1.0
        self.best_k = min_k
        
        for k in range(min_k, max_k + 1):
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            cluster_labels = km.fit_predict(self.tfidf_matrix)
            score = silhouette_score(self.tfidf_matrix, cluster_labels)
            
            self.silhouette_scores[k] = score
            self.inertias[k] = km.inertia_
            
            print(f"  -> k = {k:2d} | Silhouette Score = {score:.4f} | Inertia = {km.inertia_:.2f}")
            
            if score > best_score:
                best_score = score
                self.best_k = k
                
        print(f"[Agent 1] Optimal cluster count selected: k = {self.best_k} (Silhouette Score = {best_score:.4f})")

    def fit_final_model(self):
        """Fits KMeans using the optimal cluster count k and labels topics."""
        print(f"[Agent 1] Fitting final KMeans model with k = {self.best_k}...")
        self.kmeans_model = KMeans(n_clusters=self.best_k, random_state=42, n_init=10)
        labels = self.kmeans_model.fit_predict(self.tfidf_matrix)
        self.df['topic_cluster'] = labels
        
        # Identify top keywords per cluster center
        feature_names = np.array(self.vectorizer.get_feature_names_out())
        for i in range(self.best_k):
            center = self.kmeans_model.cluster_centers_[i]
            top_indices = center.argsort()[-5:][::-1]
            top_terms = feature_names[top_indices]
            label_str = f"Topic {i+1}: " + " / ".join(top_terms)
            self.topic_labels[i] = label_str
            print(f"  Cluster {i} -> {label_str}")
            
        self.df['topic_label'] = self.df['topic_cluster'].map(self.topic_labels)

    def plot_evaluation_metrics(self, output_img="agent1_evaluation_plots.png"):
        """Generates publication-quality diagnostic evaluation plots."""
        print(f"[Agent 1] Generating diagnostic evaluation plots -> {output_img}")
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        sns.set_theme(style="whitegrid")
        
        ks = list(self.silhouette_scores.keys())
        scores = list(self.silhouette_scores.values())
        inertias = list(self.inertias.values())
        
        # 1. Silhouette Score Curve
        axes[0].plot(ks, scores, 'o-', color='#1f77b4', linewidth=2, markersize=8)
        axes[0].axvline(x=self.best_k, color='red', linestyle='--', label=f'Optimal k={self.best_k}')
        axes[0].set_title('Silhouette Score Sweep (AIMA Ch. 20)', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Cluster Count (k)', fontsize=11)
        axes[0].set_ylabel('Silhouette Coefficient', fontsize=11)
        axes[0].legend()
        
        # 2. Elbow Method Inertia Curve
        axes[1].plot(ks, inertias, 's-', color='#2ca02c', linewidth=2, markersize=8)
        axes[1].axvline(x=self.best_k, color='red', linestyle='--', label=f'Selected k={self.best_k}')
        axes[1].set_title('Elbow Curve / Inertia (Xiao Ch. 8)', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Cluster Count (k)', fontsize=11)
        axes[1].set_ylabel('Inertia (Sum of Squared Distances)', fontsize=11)
        axes[1].legend()
        
        # 3. Topic Cluster Distribution
        cluster_counts = self.df['topic_cluster'].value_counts().sort_index()
        bars = axes[2].bar([f"T{i+1}" for i in cluster_counts.index], cluster_counts.values, color='#ff7f0e', edgecolor='black')
        axes[2].set_title('Paper Count per Topic Cluster', fontsize=12, fontweight='bold')
        axes[2].set_xlabel('Topic Cluster', fontsize=11)
        axes[2].set_ylabel('Number of Papers', fontsize=11)
        for bar in bars:
            height = bar.get_height()
            axes[2].annotate(f'{height}',
                             xy=(bar.get_x() + bar.get_width() / 2, height),
                             xytext=(0, 3),  # 3 points vertical offset
                             textcoords="offset points",
                             ha='center', va='bottom', fontweight='bold')
            
        plt.tight_layout()
        plt.savefig(output_img, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"[Agent 1] Diagnostic plot saved successfully.")

    def export_processed_data(self, csv_path="agent1_processed_catalysts.csv", 
                             xlsx_path="agent1_processed_catalysts.xlsx", 
                             json_path="agent1_processed_catalysts.json"):
        """Exports enriched dataset with cluster IDs, topic labels, and material/reaction tags."""
        print("[Agent 1] Exporting enriched datasets...")
        self.df.to_csv(csv_path, index=False, encoding='utf-8')
        self.df.to_excel(xlsx_path, index=False)
        self.df.to_json(json_path, orient='records', indent=2)
        print(f"  -> Processed CSV: {csv_path}")
        print(f"  -> Processed Excel: {xlsx_path}")
        print(f"  -> Processed JSON: {json_path}")

    def run_pipeline(self):
        """Executes full Agent 1 pipeline end-to-end."""
        self.load_data()
        self.preprocess_and_vectorize()
        self.sweep_hyperparameters(min_k=2, max_k=8)
        self.fit_final_model()
        self.plot_evaluation_metrics()
        self.export_processed_data()
        print("\n[Agent 1] Pipeline execution completed successfully!")

if __name__ == "__main__":
    agent = Agent1TopicModeling(data_path="openalex_catalysts_3000.csv")
    agent.run_pipeline()


[Agent 1] Loading dataset from: openalex_catalysts_3000.csv
[Agent 1] Successfully loaded 3000 records.
[Agent 1] Extracting domain entities and constructing TF-IDF vector matrix...
[Agent 1] Feature matrix constructed with shape: (3000, 1000)
[Agent 1] Running hyperparameter sweep over k in [2, 8]...
  -> k =  2 | Silhouette Score = 0.0163 | Inertia = 2727.09
  -> k =  3 | Silhouette Score = 0.0205 | Inertia = 2685.05
  -> k =  4 | Silhouette Score = 0.0217 | Inertia = 2658.46
  -> k =  5 | Silhouette Score = 0.0188 | Inertia = 2636.87
  -> k =  6 | Silhouette Score = 0.0206 | Inertia = 2617.02
  -> k =  7 | Silhouette Score = 0.0184 | Inertia = 2598.16
  -> k =  8 | Silhouette Score = 0.0194 | Inertia = 2581.39
[Agent 1] Optimal cluster count selected: k = 4 (Silhouette Score = 0.0217)
[Agent 1] Fitting final KMeans model with k = 4...
  Cluster 0 -> Topic 1: atom / single / single atom / sacs / fe
  Cluster 1 -> Topic 2: hydrogenation / metal / hydrogen / catalysis / based
  Cluster